# 00 · Quickstart: a muon and an electron in Super-Kamiokande

lumen follows optical photons (the light a charged particle makes in water) through a detector and tells you how
much light each light sensor (here a photomultiplier tube, PMT) sees, and when. You need no knowledge of derivatives
for this notebook: we simulate two events and look at them.

What we use:

- **Geant4 steps**: Geant4 (the standard particle-physics simulation) tracked a 600 MeV muon and a 600 MeV electron
  through Super-K's water and wrote down every short piece of each charged track (a *step*: where it starts and
  ends, how fast the particle goes). lumen turns those steps into Cherenkov light (the cone of light a particle
  faster than light in water gives off) and tracks the light.
- **A detector file**: Super-K's geometry and optics, converted once from WCSim's GDML (the geometry file Geant4
  reads). Notebook 01 shows the conversion; here it just happens if the file is missing.

Units everywhere: metres, nanoseconds, nanometres, electronvolts.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import lumen
import lumen_wc
from lumen.compare import event, references
from lumen.tools import plot

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))  # where the converted detector goes
WORK.mkdir(parents=True, exist_ok=True)

path = WORK / "sk.lumen"
if not path.exists():
    lumen.convert("lumen_wc:sk", out=path, data=references.store())  # once, about a minute
det = lumen.load(path)
det

## The two Geant4 events

Each event file holds the steps of one event (and the hits Geant4 recorded, which notebook 02 compares with).
`references.product` finds the files and checks they are the ones this package expects.

In [ ]:
events = references.product("g4ref-events-wc-933e631d", extra=[lumen_wc.REFERENCES])
muon = event.read(events / "sk_mu600.npz")
electron = event.read(events / "sk_e600.npz")
for name, ev in (("muon", muon), ("electron", electron)):
    print(f"{name}: {ev.meta['particle']} of {ev.meta['kinetic_energy_mev']} MeV from {ev.meta['position_mm']} mm "
          f"along {ev.meta['direction']}: {len(ev.steps['edep'])} steps, "
          f"{ev.steps['photons']['cherenkov'].sum():,} Cherenkov photons made by Geant4")

## Simulate

`event.simulate` makes one light source from the steps and runs `lumen.simulate` on it. `samples` is how many
Monte Carlo photons lumen follows to estimate the light; the physics (how many photons the muon really makes) comes
from the steps. More samples only make the estimate less noisy. The first call compiles the photon loop for this
detector (about a minute); later calls take a fraction of a second.

In [ ]:
light_mu = event.simulate(det, muon, samples=2**18, seed=0)
light_e = event.simulate(det, electron, samples=2**18, seed=0)

print("photons reaching PMTs: muon", round(float(light_mu.arrivals.sum())), " electron",
      round(float(light_e.arrivals.sum())))
lost = {k: float(np.sum(v)) for k, v in light_mu.health._asdict().items()}
print("photons lost to anything but absorption or detection:", lost)

## Charge and time per PMT

- `light.arrivals[i]`: expected photons reaching PMT *i* (before its quantum efficiency, the chance a photon makes
  a photoelectron);
- `lumen.readout.pe(light, params)`: expected **photoelectrons** (the PMT's charge) after that efficiency;
- `light.mean_arrival_time[i]`: the average time the light reaches PMT *i*, in ns after the particle starts.

In [ ]:
pe_mu = lumen.readout.pe(light_mu, det.params())
pe_e = lumen.readout.pe(light_e, det.params())
brightest = np.argsort(-np.asarray(pe_mu))[:5]
print(" PMT   photoelectrons   mean time (ns)")
for i in brightest:
    print(f"{i:5d} {float(pe_mu[i]):12.1f} {float(light_mu.mean_arrival_time[i]):14.1f}")
print(f"total: muon {float(pe_mu.sum()):.0f} PE, electron {float(pe_e.sum()):.0f} PE")

## Event displays

Each dot is a PMT, drawn on the tank unrolled: left to right is the angle around the tank's axis, and bottom to top
runs from the bottom cap through the barrel to the top cap. The muon makes a sharp ring (one straight track); the
electron's ring is fuzzier, because it showers into many short tracks pointing in slightly different directions.

In [ ]:
plot.event(det, pe_mu, light_mu.mean_arrival_time, title="muon")
plot.event(det, pe_e, light_e.mean_arrival_time, title="electron")
plt.show()

## How fast?

Now that the loop is compiled, time one event at two photon counts. The noise of each PMT's estimate falls as
1/sqrt(samples): four times the samples, half the noise. (Each error is estimated from the photons' own spread, each
sensor's per-photon second moment: on a sensor with tens of photons or more it is known to about 15 %.)

In [ ]:
import time

for n in (2**16, 2**18):
    event.simulate(det, muon, samples=n, seed=1)  # a new photon count compiles once: time the second call
    t0 = time.perf_counter()
    light = event.simulate(det, muon, samples=n, seed=2)
    light.arrivals.block_until_ready()
    i = int(np.argmax(np.asarray(light.arrivals)))
    rel = float(light.arrivals_error[i] / light.arrivals[i])
    print(f"{n:7d} samples: {time.perf_counter() - t0:.2f} s; brightest PMT's relative error {rel:.3f}")

**Next:** 01 converts the detector yourself; 02 checks lumen against Geant4 and WCSim; 03 explains what a
derivative of a simulation is; 04 uses derivatives to calibrate the water.